# GET BOOKING DETAILS

## **Summary**
This JSON is a student record containing personal details, course enrollments with schedules, and basic school information. It helps to manage student bookings, monitor class schedules, and identify additional services. The record also reserves placeholders for accommodations, transfers, and invoices, which are empty in this case.
## **Top-Level Structure**
- **`data`**: The main container for all information in this record.

---

## **Student Information**
- **`firstname`**: The student's first name (`Ana`).
- **`surname`**: The student's last name (`Sanchez Hoyos`).
- **`email`**: The student's email address (`anasanchezh4@gmail.com`).
- **`number`**: The unique student number (`SLC01642`).

**Purpose**: Identifies the student.

---

## **Booking Information**0
### **Courses**
Each course has:
- **`from`**: Start date of the course.
- **`until`**: End date of the course.
- **`weeks`**: Duration of the course in weeks.
- **`name`**: Course name (e.g., "Evening Classes (5.5hrs/wk) - DO NOT USE").
- **`category`**: Type/category of the course (e.g., "Evening Classes").
- **`note`**: Additional notes about the course.
- **`flexible_allocation`**: Indicates flexibility in class assignment (`1` for yes, `0` for no).
- **`custom_fields`**: Reserved for extra custom data (empty here).
- **`additional_services`**: Reserved for supplementary services (empty here).
- **`assignments`**: Details about assigned classes during the course:
  - **`week`**: Week in which the class occurs.
  - **`class`**: Class name (e.g., "FCE").
  - **`days`**: Days of the week (numerical values, e.g., `1` for Monday, `3` for Wednesday).
  - **`start`**: Class start time.
  - **`end`**: Class end time.
  - **`teacher`**: Teacher assigned (empty here).

**Purpose**: Provides complete details of the courses a student is enrolled in, including schedule and assignments.

---

### **Accommodations**
- **`accommodations`**: Empty list indicating no accommodations booked for the student.

### **Transfers**
- **`transfers`**: Empty list indicating no transfer services are associated.

**Purpose**: Tracks additional services like accommodation or transfers if provided.

---

## **Invoices**
- **`invoices`**: Empty list indicating no invoice records associated with the student.

**Purpose**: Would record financial transactions for the student.

---

## **School Information**
- **`name`**: School name (`Example School`).
- **`address`**: Street address (`1 Brunswick Place`).
- **`zip`**: Postal code (`SO15 2AN`).
- **`city`**: City (`ExampleCity`).
- **`country_id`**: Country code (`GB` for the UK).
- **`phone`**: Contact number (`02380231397`).
- **`email`**: School's email address (`school@example.com`).

**Purpose**: Identifies the institution providing the courses.

---




In [ ]:
import requests
import json
import os

# Replace with your actual API URL and Token
BASE_URL = "https://your-school.example.com/api/1.0/ts/booking/"
API_TOKEN = os.environ["API_TOKEN"]  # Replace with your actual Bearer Token

# Define the range of booking IDs to query
booking_ids = range(35933, 35934)  # Replace with your desired range of IDs

# Headers for the request
headers = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}

# Paths for output files
output_dir = "school-api/Get Booking Details/"
os.makedirs(output_dir, exist_ok=True)  # Ensure the directory exists

bookings_file_path = os.path.join(output_dir, "bookings.json")
status_file_path = os.path.join(output_dir, "booking_status.json")


# Container for storing fetched booking details
bookings = []
# Container for booking statuses
status_log = []

# Loop through booking IDs and fetch details
for booking_id in booking_ids:
    try:
        # Construct the URL with the booking ID
        url = f"{BASE_URL}{booking_id}?include_inactive_services=1"

        # Make the GET request
        response = requests.get(url, headers=headers)

        # Check the response status
        if response.status_code == 200:
            data = response.json()
            data["booking_id"] = booking_id  # Add the booking ID to the data
            bookings.append(data)
            status_log.append({"booking_id": booking_id, "status": 1})
            print(f"Successfully fetched booking {booking_id}")
        elif response.status_code == 404:
            print(f"Booking {booking_id} not found.")
            status_log.append({"booking_id": booking_id, "status": 0})
        else:
            print(f"Error {response.status_code} for booking {booking_id}: {response.text}")
            status_log.append({"booking_id": booking_id, "status": 0})
    except Exception as e:
        print(f"Error fetching booking {booking_id}: {e}")
        status_log.append({"booking_id": booking_id, "status": 0})

# Save fetched bookings to a file
with open(bookings_file_path, "w") as file:
    json.dump(bookings, file, indent=4)

# Save the booking statuses to a separate file
with open(status_file_path, "w") as file:
    json.dump(status_log, file, indent=4)

print(f"Fetched {len(bookings)} bookings and saved to {bookings_file_path}.")
print(f"Booking statuses saved to {status_file_path}.")